# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [20]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path("E Commerce Dataset.xlsx"),
    Path("../data/E Commerce Dataset.xlsx"),
    
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请修改 DATA_PATH。")

root_candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "notebooks").exists()),
    Path.cwd()
)
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据：{DATA_PATH}")
print(f"项目输出目录：{OUTPUT_DIR}")
print(f"原始数据形状：{raw_df.shape}")
raw_df.head()

原始数据：..\data\E Commerce Dataset.xlsx
项目输出目录：D:\desk\holiday study\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project
原始数据形状：(5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [21]:
print(Path.cwd())

D:\desk\holiday study\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\notebooks


In [22]:
# 在此写下你的答案：
# 1.每条记录代表单个电商客户的相关信息
# 2.churn
# 3.CustomerID只是客户唯一编号，不存在数值大小意义

---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [23]:
def build_quality_report(data):
    """返回字段级数据质量报告。"""
    # TODO：返回一个 DataFrame，至少包含：
    # 数据类型、缺失数量、缺失比例(%)、唯一值数量
    report = pd.DataFrame({
        "数据类型":data.dtypes,
        "缺失数量":data.isna().sum(),
        "缺失比例(%)":round((data.isna().sum() / data.shape[0]) * 100, 2),
        "唯一值数量":data.nunique()
    })
    report = report.sort_values("缺失比例(%)", ascending=False)
    return report


# TODO：生成清洗前质量报告
quality_before = build_quality_report(raw_df)
display(quality_before)

,数据类型,缺失数量,缺失比例(%),唯一值数量
DaySinceLastOrder,float64,307,5.45,22
OrderAmountHikeFromlastYear,float64,265,4.71,16
Tenure,float64,264,4.69,36
OrderCount,float64,258,4.58,16
CouponUsed,float64,256,4.55,17
HourSpendOnApp,float64,255,4.53,6
WarehouseToHome,float64,251,4.46,34
CustomerID,int64,0,0.00,5630
PreferredLoginDevice,str,0,0.00,3
Churn,int64,0,0.00,2


### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [24]:
# TODO：完成项目初始审计
print("完全重复行数：",raw_df.duplicated().sum())
print("CustomerID 重复数量：",raw_df["CustomerID"].duplicated().sum())

#- Churn 的频数和流失率；
print(raw_df["Churn"].value_counts())
#计算流失率
print("流失率：",raw_df["Churn"].mean().round(4))

#- 主要类别字段的频数。
for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"\n{col}")
    print(raw_df[col].value_counts())

完全重复行数： 0
CustomerID 重复数量： 0
Churn
0    4682
1     948
Name: count, dtype: int64
流失率： 0.1684

PreferredLoginDevice
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

PreferredPaymentMode
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

PreferedOrderCat
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [25]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [26]:
def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。

    参数：
        data: 原始用户行为 DataFrame

    返回：
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """
    # TODO：复制数据，避免覆盖原始数据
    cleaned_df = data.copy()
    # TODO：创建日志列表 logs
    logs = []
    # TODO：删除完全重复行，并记录日志
    before_cnt = len(cleaned_df)
    cleaned_df = cleaned_df.drop_duplicates()
    after_cnt = len(cleaned_df)
    affected = before_cnt - after_cnt
    logs.append({
        "处理步骤": "删除完全重复行",
        "处理规则": "删除全部完全重复记录",
        "处理前记录数": before_cnt,
        "处理后记录数": after_cnt,
        "影响记录数": affected
    })
    # TODO：对 NUMERIC_MISSING_COLS 使用中位数填补，并记录每列影响数量
    for col in NUMERIC_MISSING_COLS:
        before_cnt_col = len(cleaned_df)
        missing_count = cleaned_df[col].isna().sum()
        med_val = cleaned_df[col].median()
        cleaned_df[col] = cleaned_df[col].fillna(med_val)
        logs.append({
            "处理步骤": f"缺失值填补-{col}",
            "处理规则": "数值字段缺失使用总体中位数填补",
            "处理前记录数": before_cnt_col,
            "处理后记录数": before_cnt_col,
            "影响记录数": missing_count
        })
    # TODO：对 CATEGORY_MAPPINGS 完成类别标准化，并记录每条映射影响数量
    for col , map_dicts in CATEGORY_MAPPINGS.items():
        for old_val,new_val in map_dicts.items():
            affected_num = (cleaned_df[col] == old_val).sum()
            cleaned_df[col] = cleaned_df[col].replace(old_val,new_val)
            logs.append({
                "处理步骤": f"类别标准化-{col}:{old_val}→{new_val}",
                "处理规则": "同义类别统一名称",
                "处理前记录数": len(cleaned_df),
                "处理后记录数": len(cleaned_df),
                "影响记录数": affected_num
            })
    # TODO：将 Churn 和 Complain 转为整数类型
    for type_col in ["Churn","Complain"]:
        cleaned_df[type_col] = cleaned_df[type_col].astype(int)
        logs.append({
            "处理步骤": f"类型转换-{type_col}",
            "处理规则": "转为int整数类型",
            "处理前记录数": len(cleaned_df),
            "处理后记录数": len(cleaned_df),
            "影响记录数": 0
        })
    # TODO：返回 cleaned_df 与 cleaning_log
    cleaning_log = pd.DataFrame(logs)
    return cleaned_df,cleaning_log


### 任务 3：运行清洗函数并查看日志

In [27]:
# TODO：执行清洗
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)
#
display(cleaning_log)
cleaned_df.head()

,处理步骤,处理规则,处理前记录数,处理后记录数,影响记录数
0,删除完全重复行,删除全部完全重复记录,5630,5630,0
1,缺失值填补-Tenure,数值字段缺失使用总体中位数填补,5630,5630,264
2,缺失值填补-WarehouseToHome,数值字段缺失使用总体中位数填补,5630,5630,251
3,缺失值填补-HourSpendOnApp,数值字段缺失使用总体中位数填补,5630,5630,255
4,缺失值填补-OrderAmountHikeFromlastYear,数值字段缺失使用总体中位数填补,5630,5630,265
5,缺失值填补-CouponUsed,数值字段缺失使用总体中位数填补,5630,5630,256
6,缺失值填补-OrderCount,数值字段缺失使用总体中位数填补,5630,5630,258
7,缺失值填补-DaySinceLastOrder,数值字段缺失使用总体中位数填补,5630,5630,307
8,类别标准化-PreferredLoginDevice:Phone→Mobile Phone,同义类别统一名称,5630,5630,1231
9,类别标准化-PreferredPaymentMode:COD→Cash on Delivery,同义类别统一名称,5630,5630,365


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [28]:
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }

# TODO：构建 tenure_bins、tenure_labels，并用 pd.cut 新建 TenureGroup
tenure_bins = [0,12,24,36,48,60,float("inf")]
tenure_lables = ["0-12月","13-24月","25-36月","37-48月","49-60月","60月以上"]
cleaned_df["TenureGroup"] = pd.cut(cleaned_df["Tenure"],tenure_bins,labels=tenure_lables,right=False)
# TODO：新建 IsMobileLogin，移动端为 1，其他设备为 0
cleaned_df["IsMobileLogin"]= (cleaned_df["PreferredLoginDevice"] == "Mobile Phone").astype(int)
# TODO：生成 outlier_report（每行对应一个待检查字段）
check_cols = ["WarehouseToHome","OrderCount", "CashbackAmount"]
outlier_report = pd.DataFrame([iqr_outlier_summary(cleaned_df[col]) for col in check_cols], index=check_cols)
display(outlier_report)

,Q1,Q3,下限,上限,候选异常值数量
WarehouseToHome,9.00,20.00,-7.50,36.50,2
OrderCount,1.00,3.00,-2.00,6.00,703
CashbackAmount,145.77,196.39,69.84,272.33,438


### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [29]:
# TODO：完成业务规则检查
rule_1 = (cleaned_df["Tenure"] < 0).sum()                  # 使用时长小于 0
rule_2 = (cleaned_df["WarehouseToHome"] < 0).sum()         # 仓库距离小于 0
rule_3 = (cleaned_df["OrderCount"] <= 0).sum()            # 订单数小于或等于 0
rule_4 = (cleaned_df["CashbackAmount"] < 0).sum()          # 返现金额小于 0

business_rule_report = pd.DataFrame({
    "规则": ["使用时长小于 0",
        "仓库距离小于 0",
        "订单数小于或等于 0",
        "返现金额小于 0"],
     "不合规记录数": [rule_1, rule_2, rule_3, rule_4]
})
display(business_rule_report)
#
# 处理结论：本次业务规则检查针对使用时长、仓库距离、订单数量、返现金额四类核心业务字段进行合法性校验。
#所有字段不合规记录数均为0，说明数据集不存在负数值、零订单等违反业务逻辑的脏数据，
#字段取值完全符合电商业务常识，无需进行异常剔除或修正处理。

,规则,不合规记录数
0,使用时长小于 0,0
1,仓库距离小于 0,0
2,订单数小于或等于 0,0
3,返现金额小于 0,0


---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [30]:
# TODO：完成最终验收
quality_after = build_quality_report(cleaned_df)
#
assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique()
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique()
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique()
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns)
#
# TODO：导出下列文件，使用 utf-8-sig 编码：
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=False, encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=False, encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")
#
# TODO：输出 outlier_report 和 business_rule_report
print("=====候选异常值汇总报告=====")
display(outlier_report)
print("\n=====业务合规检查报告=====")
display(business_rule_report)

outlier_report.to_csv(OUTPUT_DIR / "outlier_report.csv", index=False, encoding="utf-8-sig")
business_rule_report.to_csv(OUTPUT_DIR / "business_rule_report.csv", index=False, encoding="utf-8-sig")
# TODO：输出交付文件的路径
print("\n 全部文件导出完成，输出目录：")
print(OUTPUT_DIR.resolve())
print("生成文件清单：")
print("- data_quality_before.csv  清洗前数据质量报告")
print("- data_quality_after.csv   清洗后数据质量报告")
print("- cleaning_log.csv         数据清洗操作日志")
print("- ecommerce_customer_cleaned.csv 最终清洗数据集")

=====候选异常值汇总报告=====


,Q1,Q3,下限,上限,候选异常值数量
WarehouseToHome,9.00,20.00,-7.50,36.50,2
OrderCount,1.00,3.00,-2.00,6.00,703
CashbackAmount,145.77,196.39,69.84,272.33,438



=====业务合规检查报告=====


,规则,不合规记录数
0,使用时长小于 0,0
1,仓库距离小于 0,0
2,订单数小于或等于 0,0
3,返现金额小于 0,0



 全部文件导出完成，输出目录：
D:\desk\holiday study\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project
生成文件清单：
- data_quality_before.csv  清洗前数据质量报告
- data_quality_after.csv   清洗后数据质量报告
- cleaning_log.csv         数据清洗操作日志
- ecommerce_customer_cleaned.csv 最终清洗数据集


## 项目复盘

请在提交前用不超过 200 字回答：

1. 本项目发现了哪些数据质量问题？
2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？
3. 为什么清洗后的数据可以作为第五天分析的输入？
4. 哪些处理规则仍需要业务人员确认？

In [31]:
"""1.原始数据存在数值字段缺失、类别名称缩写不统一、存在 IQR 识别的候选异常值；存在完全重复样本。
2.数值缺失使用总体中位数填充；同义类别统一标准化；候选异常值仅汇总记录，不直接删除。
3.已完成去重、缺失补齐、类别统一、衍生特征构建，并通过验收断言，数据格式与取值规范，满足后续建模分析要求。
4.中位数填补方案、IQR 识别出的异常样本是否保留，仍需要业务人员结合业务场景复核确"""

'1.原始数据存在数值字段缺失、类别名称缩写不统一、存在 IQR 识别的候选异常值；存在完全重复样本。\n2.数值缺失使用总体中位数填充；同义类别统一标准化；候选异常值仅汇总记录，不直接删除。\n3.已完成去重、缺失补齐、类别统一、衍生特征构建，并通过验收断言，数据格式与取值规范，满足后续建模分析要求。\n4.中位数填补方案、IQR 识别出的异常样本是否保留，仍需要业务人员结合业务场景复核确'